In [1]:
import argparse
import logging
from typing import List, Tuple
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
cd ..

/teamspace/studios/this_studio/recsys2025


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
from baseline.aggregated_features_baseline.constants import (
    EVENT_TYPE_TO_COLUMNS,
)
from data_utils.utils import (
    load_with_properties,
)
from data_utils.data_dir import DataDir
from baseline.aggregated_features_baseline.features_aggregator import (
    FeaturesAggregator,
)

In [ ]:
logging.basicConfig()
logger = logging.getLogger(__name__)
logger.setLevel(level=logging.INFO)

In [22]:
def load_relevant_clients_ids(input_dir: Path) -> np.ndarray:
    return np.load(input_dir / "relevant_clients.npy")

In [7]:
def save_embeddings(
    embeddings_dir: Path, embeddings: np.ndarray, client_ids: np.ndarray
):
    """
    Function creates embeddings directory and saves embeddings in competition entry format.

    Args:
    embeddings_dir (Path): The directory where to save embeddings and client_ids.
    embeddings (np.ndarray): 2-d array storing embeddings.
    client_ids (np.ndarray): 1-d array storing client_ids corresponding to vectors from embeddings array.
    """
    logger.info("Saving embeddings")
    embeddings_dir.mkdir(parents=True, exist_ok=True)
    np.save(embeddings_dir / "embeddings.npy", embeddings)
    np.save(embeddings_dir / "client_ids.npy", client_ids)


def create_embeddings(
    data_dir: DataDir,
    num_days: List[int],
    top_n: int,
    relevant_client_ids: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate and merge user representation embeddings for specified event types.

    This function processes event data from CSV files, aggregates user's events based
    on specified columns for each event type, and merges these embeddings into a single
    user representation.

    Args:
        data_dir (DataDir): The DataDir class where Paths to raw event data, input and targte folders are stored.
        num_days (List[int]): A list of time windows (in days) for generating features.
        Each time window will produce different set of features from aggregated events
        from defined period.
        top_n (int): Number of columns' top values to consider for aggregating events.

    Returns:
        Tuple[np.ndarray, np.ndarray] : generated feature matrix and the list of all
        clients in two np.ndarray's.
    """
    aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )
    for event_type in EVENT_TYPE_TO_COLUMNS.keys():
        logger.info("Generating features for %s event type", event_type.value)
        logger.info("Loading data...")
        event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)
        event_df["timestamp"] = pd.to_datetime(event_df.timestamp)
        logger.info("Generating features...")
        aggregator.generate_features(
            event_type=event_type,
            client_id_column="client_id",
            df=event_df,
            columns=EVENT_TYPE_TO_COLUMNS[event_type],
        )

    logger.info("Merging features into embeddings")
    client_ids, embeddings = aggregator.merge_features()
    return client_ids, embeddings


In [8]:
def get_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data-dir",
        type=str,
        required=True,
        help="Directory with input and target data – produced by data_utils.split_data",
    )
    parser.add_argument(
        "--embeddings-dir",
        type=str,
        required=True,
        help="Directory where to store generated embeddings",
    )
    parser.add_argument(
        "--num-days",
        nargs="*",
        type=int,
        default=[1, 7, 30],
        help="Numer of days to compute features",
    )
    parser.add_argument(
        "--top-n",
        type=int,
        default=10,
        help="Number of top column values to consider in feature generation",
    )
    return parser

In [20]:
data_dir = "/teamspace/studios/this_studio/ubc_data"
data_dir = DataDir(Path(data_dir))

In [12]:
embeddings_dir = "/teamspace/studios/this_studio/ubc_data/embeddings"
embeddings_dir = Path(embeddings_dir)

In [23]:
relevant_client_ids = load_relevant_clients_ids(input_dir=data_dir.input_dir)

In [24]:
relevant_client_ids

array([ 5963217, 17797869, 18408314, ..., 22423862, 16846917, 14884775])

In [26]:
client_ids, embeddings = create_embeddings(
    data_dir=data_dir,
    num_days=[1, 7, 30],
    top_n=10,
    relevant_client_ids=relevant_client_ids,
)

INFO:__main__:Generating features for product_buy event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 416269/416269 [04:28<00:00, 1553.17it/s]
INFO:__main__:Generating features for add_to_cart event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 496756/496756 [05:29<00:00, 1509.37it/s]
INFO:__main__:Generating features for remove_from_cart event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 253428/253428 [02:49<00:00, 1496.29it/s]
INFO:__main__:Generating features for page_visit event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 720247/720247 [07:21<00:00, 1629.79it/s]
INFO:__main__:Generating features for search_query event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 247597/247597 [01:01<00:00, 4018.87it/s]
INFO:__main__:Merging features into embeddings
100%|██████████| 8

In [27]:
type(client_ids)

numpy.ndarray

In [28]:
len(client_ids)

858489

In [29]:
type(embeddings)

numpy.ndarray

In [30]:
len(embeddings)

858489

In [33]:
len(embeddings[0])

320

In [34]:
embeddings[0]

array([2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [35]:
# TODO: Debug create_embeddings.py 

# TODO: Debug aggregator.generate_features( 

In [36]:
data_dir

In [37]:
##################### DRAFT  ########################

In [ ]:

def create_embeddings(
    data_dir: DataDir,
    num_days: List[int],
    top_n: int,
    relevant_client_ids: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate and merge user representation embeddings for specified event types.

    This function processes event data from CSV files, aggregates user's events based
    on specified columns for each event type, and merges these embeddings into a single
    user representation.

    Args:
        data_dir (DataDir): The DataDir class where Paths to raw event data, input and targte folders are stored.
        num_days (List[int]): A list of time windows (in days) for generating features.
        Each time window will produce different set of features from aggregated events
        from defined period.
        top_n (int): Number of columns' top values to consider for aggregating events.

    Returns:
        Tuple[np.ndarray, np.ndarray] : generated feature matrix and the list of all
        clients in two np.ndarray's.
    """
    aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )
    for event_type in EVENT_TYPE_TO_COLUMNS.keys():
        logger.info("Generating features for %s event type", event_type.value)
        logger.info("Loading data...")
        event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)
        event_df["timestamp"] = pd.to_datetime(event_df.timestamp)
        logger.info("Generating features...")
        aggregator.generate_features(
            event_type=event_type,
            client_id_column="client_id",
            df=event_df,
            columns=EVENT_TYPE_TO_COLUMNS[event_type],
        )

    logger.info("Merging features into embeddings")
    client_ids, embeddings = aggregator.merge_features()
    return client_ids, embeddings


In [39]:
num_days=[1, 7, 30]
top_n=10

In [57]:
EVENT_TYPE_TO_COLUMNS.keys()

dict_keys([<EventTypes.PRODUCT_BUY: 'product_buy'>, <EventTypes.ADD_TO_CART: 'add_to_cart'>, <EventTypes.REMOVE_FROM_CART: 'remove_from_cart'>, <EventTypes.PAGE_VISIT: 'page_visit'>, <EventTypes.SEARCH_QUERY: 'search_query'>])

In [72]:
i = 0 

for event_type in EVENT_TYPE_TO_COLUMNS.keys():

    print(event_type)
    if i == 0: 
        break
    i = i + 1

EventTypes.PRODUCT_BUY


In [73]:
event_type

<EventTypes.PRODUCT_BUY: 'product_buy'>

In [74]:
event_type.value

'product_buy'

In [75]:
aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )

In [76]:
event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)

In [77]:
event_df

,client_id,timestamp,sku,category,price,name
0,17649961,2022-07-23 20:15:25,18485,5492,72,[187 47 120 237 234 172 91 172 67 153 32 ...
1,16696114,2022-07-11 16:31:30,81192,6519,99,[241 241 241 241 241 241 241 241 112 241 241 2...
2,10238779,2022-05-29 19:35:40,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
3,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
4,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
...,...,...,...,...,...,...
1315056,7535041,2022-08-19 04:58:40,269962,2895,14,[128 159 53 89 131 143 53 171 53 214 141 1...
1315057,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315058,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315059,20304565,2022-09-02 12:38:50,79597,5349,0,[196 97 71 127 98 174 234 70 25 2 202 2...


In [78]:

event_df["timestamp"] = pd.to_datetime(event_df.timestamp)

In [ ]:
# TODO: Debuggear generate_features dentro de FeaturesAggregator 